In [1]:
# Updated installation snippet
!pip install -q \
    langchain \
    langchain-core \
    langchain-community \
    langchain-google-genai \
    faiss-cpu \
    rank_bm25

print("[System] Hybrid environment installed successfully with latest compatible versions.")

[System] Hybrid environment installed successfully with latest compatible versions.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from pathlib import Path
from dotenv import load_dotenv
import os

c:\Users\Public\Documents\ai-foundations-lab\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\lenovo\AppData\Local\Temp\ipykernel_13564\3345776311.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# Load .env (development) into env vars; production should set real env vars or use a secrets manager.
load_dotenv()  # reads .env if present

# Prefer explicit env var; fall back to a secrets file only if provided
api_key = os.getenv("GEMINI_API_KEY") 

if not api_key:
    secrets_path = Path(os.getenv("SECRETS_PATH", Path("secrets") / "api"))
    if secrets_path.exists():
        with secrets_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("GEMINI_API_KEY="):
                    api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in environment or secrets file. Set GEMINI_API_KEY or SECRETS_PATH.")

os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded:", bool(api_key))



API key loaded: True


In [4]:
# Initialize the Embedder for Dense Engine

embedder = GoogleGenerativeAIEmbeddings(model = "models/gemini-embedding-001")

#1. Dataset (Executive Business News)
mock_news_data = [
    "Q4 Financial Overview: Income pushbacks observed due to late client payments.",
    "URGENT: Project Alpha budget overrun flagged under Invoice #INV-2023-99X.",
    "IT Support Ticket: The email server is experiencing delays routing the budget.",
    "General HR Update: Remote work policies extended through Q4."
]


# Convert strings to Langchain Objects 
docs = [Document(page_content=text) for text in mock_news_data]


In [7]:
#Initializing the Dual Engines 
print("[System] Initializing Dense Vector Engine (FAISS)...")

# Engina A: The Semantic Profiler 
# We embed the documents into the 768 dimensional hypersphere
vectorstore = FAISS.from_documents(docs, embedder)
faiss_retriever = vectorstore.as_retriever(search_kwargs = {"k":2})

#Engine B : The Fingerprint Analyst 
# We build a local statistical index based on term frequency
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2

print("[System] Both engines online and awaiting fusion.")

[System] Initializing Dense Vector Engine (FAISS)...
[System] Both engines online and awaiting fusion.


In [8]:
# The Reciprocal Rank Fusion Execution 
print("\n[System] Fusing Engines via Reciprocal Rank Fusion (RRF)...")

# The weights parameter allows us to bias the RRF algorithm 
# Setting it to [0.5,0.5] weights the Dense and Sparse engines equally.
ensembleretriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights = [0.5,0.5]
)


print("[System] Hybrid Search Online. Commencing Stress Test...\n")

hybrid_query = "Find the exact budget overrun details for Invoice #INV-2023-99X."

print(f"User Query: '{hybrid_query}'\n")

# The single invoke() call triggers both databases and runs the math
fused_results = ensembleretriever.invoke(hybrid_query)

print("=" * 40)
print(" RRF FINAL RANKINGS ")
print("="* 40)


for i, doc in enumerate(fused_results):
    print(f"\n---Rank{i+1}---")
    print(doc.page_content)



[System] Fusing Engines via Reciprocal Rank Fusion (RRF)...
[System] Hybrid Search Online. Commencing Stress Test...

User Query: 'Find the exact budget overrun details for Invoice #INV-2023-99X.'

 RRF FINAL RANKINGS 

---Rank1---
URGENT: Project Alpha budget overrun flagged under Invoice #INV-2023-99X.

---Rank2---
IT Support Ticket: The email server is experiencing delays routing the budget.
